# MiniMax H3（動画）を Google Colab で動かす
**使い方**：セルの左の ▶ を、**上から順に1つずつ**押します（「すべてのセルを実行」は使わない）。各セルの最後に **`OK:`** か **`NG:`** が出ます。`NG:` のときは、書いてあるとおりにしてから**同じセルをもう一度**押します。
- GPU は **G4**（なければ A100）を選んでおく：メニュー「ランタイム」→「ランタイムのタイプを変更」。
- セル5 で出る URL（trycloudflare.com）は**クリックせず、コピーして新しいタブのアドレス欄に貼る**。人には教えない。
- 使い終わったら、最後のセルで**必ずランタイムを止める**（つないでいる間はユニットが減り続ける）。
- MiniMax H3 の使い方の条件（ライセンス）は、使う前に自分で確かめてください。

In [ ]:
# 0. Google ドライブにつなぐ（できた動画をドライブに残すため）。許可の窓が出たら「Google ドライブに接続」→ 許可
from google.colab import drive
drive.mount('/content/drive')
import os
BASE = '/content/drive/MyDrive/AI_Colab/H3'
os.makedirs(BASE + '/inputs', exist_ok=True)
os.makedirs(BASE + '/output', exist_ok=True)
print('OK: ドライブにつながった。素材は マイドライブ/AI_Colab/H3/inputs、できた動画は マイドライブ/AI_Colab/H3/output に入ります')

In [ ]:
# 1. 借りた GPU を確かめる（足りなければ、ここで止める）
import time, subprocess, shutil, json, re, os
T0 = time.time()
def ok(msg): print('OK:', msg)
def ng(msg, todo): print('NG:', msg, '\n   → やること:', todo)
RESULT = {'ready': False}
smi = subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout
q = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
if not q:
    ng('GPU につながっていない', 'メニュー「ランタイム」→「ランタイムのタイプを変更」→ G4（なければ A100）→ 保存。そのあと、このセルからやり直す')
else:
    RESULT['gpu'], RESULT['vram'], RESULT['driver'] = [x.strip() for x in q.split(',')]
    m = re.search(r'CUDA Version:\s*([\d.]+)', smi)
    vram_gb = int(re.sub(r'\D', '', RESULT['vram']) or 0) / 1024
    disk = shutil.disk_usage('/content').free / 2**30
    if vram_gb < 70:
        ng(f"GPU のメモリが {vram_gb:.0f}GB（H3 は 70GB ほど要る）", 'ランタイムを接続解除して削除し、G4 か A100 を選び直す（T4・L4 では足りない）')
    elif disk < 80:
        ng(f'ディスクの空きが {disk:.0f}GB（80GB 以上要る）', 'ランタイムを接続解除して削除し、G4 か A100 でつなぎ直す')
    elif m and float(m.group(1)) < 13.0:
        ng(f'GPU のドライバが古い（CUDA {m.group(1)}）', 'ランタイムを接続解除して削除し、つなぎ直す（別の GPU を試す）')
    else:
        RESULT['ready'] = True
        ok(f"{RESULT['gpu']}・メモリ {vram_gb:.0f}GB・ディスク空き {disk:.0f}GB")

In [ ]:
# 2. ComfyUI と PyTorch を入れる（2〜5分。途中の赤い文字は、最後に OK なら気にしない）
assert RESULT.get('ready'), 'NG: セル1が OK になっていない。セル1から順に押す'
t = time.time()
WHL = 'nvidia_cublas-13.1.1.3-py3-none-manylinux_2_27_x86_64.whl'
!test -d /content/ComfyUI || git clone -q https://github.com/Comfy-Org/ComfyUI /content/ComfyUI
!pip -q install -r /content/ComfyUI/requirements.txt
for _ in range(10):
    subprocess.run(['curl', '-sL', '-C', '-', '--retry', '20', '--retry-all-errors', '-o', f'/tmp/{WHL}', 'https://pypi.nvidia.com/nvidia-cublas/' + WHL])
    if os.path.exists(f'/tmp/{WHL}') and os.path.getsize(f'/tmp/{WHL}') >= 420_000_000:
        break
    time.sleep(3)
subprocess.run(['pip', 'install', '-q', f'/tmp/{WHL}'])
!pip -q install --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu130
!pip -q install "huggingface_hub[hf_xet]"
def cublas_version():
    mm = re.search(r'Version:\s*(\S+)', subprocess.run(['pip', 'show', 'nvidia-cublas'], capture_output=True, text=True).stdout)
    return mm.group(1) if mm else ''
if not cublas_version().startswith('13.1'):
    subprocess.run(['pip', 'install', '-q', '--no-deps', '--force-reinstall', f'/tmp/{WHL}'])
import torch
if os.path.exists('/content/ComfyUI/main.py') and torch.cuda.is_available() and 'cu13' in torch.__version__ and cublas_version().startswith('13.1'):
    ok(f'ComfyUI と PyTorch {torch.__version__} を入れた（{round(time.time() - t)}秒）')
else:
    ng('ComfyUI か PyTorch がそろっていない', 'メニュー「ランタイム」→「セッションを再起動」→ セル1 → このセル の順にやり直す')

In [ ]:
# 3. H3 のモデル4本（合計 約54GB）を入れる（2〜5分）。途中で止まったら、もう一度押せば続きから
assert RESULT.get('ready'), 'NG: セル1が OK になっていない。セル1から順に押す'
os.environ['HF_XET_HIGH_PERFORMANCE'] = '1'
from huggingface_hub import hf_hub_download
M = '/content/ComfyUI/models'
FILES = [('diffusion_models', 'minimax_h3_fastvideo_vsa_datafree_1300step_4step_int8_convrot.safetensors', 'Kijai/MiniMax-H3-experimental', 'minimax_h3_fastvideo_vsa_datafree_1300step_4step_int8_convrot.safetensors', 22.9),
         ('vae', 'minimax_h3_video_vae_int8_convrot.safetensors', 'Kijai/MiniMax-H3-experimental', 'minimax_h3_video_vae_int8_convrot.safetensors', 3.17),
         ('text_encoders', 'qwen3vl_32b_minimax_h3_int8_convrot.safetensors', 'Comfy-Org/MiniMax-H3', 'text_encoders/qwen3vl_32b_minimax_h3_int8_convrot.safetensors', 27.14),
         ('vae', 'minimax_h3_audio_vae_fp32.safetensors', 'Comfy-Org/MiniMax-H3', 'vae/minimax_h3_audio_vae_fp32.safetensors', 0.61)]
t = time.time()
for sub, name, repo, path, want in FILES:
    dst = f'{M}/{sub}/{name}'
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if os.path.exists(dst) and os.path.getsize(dst) / 1e9 > want * 0.95:
        print('  ', name, 'はもう入っている'); continue
    try:
        shutil.move(hf_hub_download(repo, path, local_dir='/content/hf_tmp'), dst)
        print('  ', name, round(os.path.getsize(dst) / 1e9, 1), 'GB')
    except Exception as e:
        print('  NG:', name, str(e)[:150])
short = [n for s, n, _, _, w in FILES if not os.path.exists(f'{M}/{s}/{n}') or os.path.getsize(f'{M}/{s}/{n}') / 1e9 < w * 0.95]
if short:
    ng('入っていないモデル：' + ', '.join(short), 'このセルをもう一度押す（続きから入る）')
else:
    ok(f'モデル4本がそろった（{round(time.time() - t)}秒）')

In [ ]:
# 4. ワークフロー・プログラム・見本の素材を取ってくる（AI is in Wonderland の公開リポジトリから）
assert RESULT.get('ready'), 'NG: セル1が OK になっていない。セル1から順に押す'
import urllib.request
RAW = 'https://raw.githubusercontent.com/ai-is-in-wonderland/colab-minimax-h3/main'
WFD = '/content/ComfyUI/user/default/workflows'
os.makedirs(WFD, exist_ok=True)
try:
    urllib.request.urlretrieve(RAW + '/workflows/h3_t2v_minimal_864x480.json', WFD + '/test_t2v_5s.json')
    urllib.request.urlretrieve(RAW + '/workflows/h3_i2v_lipsync_1536x864.json', '/content/h3_i2v_lipsync.json')
    urllib.request.urlretrieve(RAW + '/h3_job_patch.py', '/content/h3_job_patch.py')
    S = BASE + '/inputs/sample/c01'
    if not os.path.exists(S + '/start.png'):
        os.makedirs(S, exist_ok=True)
        for f in ('start.png', 'voice.wav', 'prompt.txt'):
            urllib.request.urlretrieve(RAW + '/sample/c01/' + f, S + '/' + f)
    ok('テスト用のワークフロー（test_t2v_5s）を置いた。見本の素材は マイドライブ/AI_Colab/H3/inputs/sample/c01')
except Exception as e:
    ng('取ってこられなかった（' + str(e)[:120] + '）', '少し待って、このセルをもう一度押す')

In [ ]:
# 5. ComfyUI を起動して、開くための URL を作る（1〜3分）。できた動画は自動でドライブにコピーされる
assert RESULT.get('ready'), 'NG: セル1が OK になっていない。セル1から順に押す'
!wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /content/cloudflared
import glob, threading
SYNC_SRC = '/content/ComfyUI/output'
SYNC_DST = BASE + '/output/' + time.strftime('%Y%m%d_%H%M')   # つなぐたびに新しいフォルダ（前の動画を上書きしない）
def autosync():   # 書き終わった mp4 を5秒ごとにドライブへコピー
    seen = {}
    while True:
        try:
            for f in glob.glob(SYNC_SRC + '/**/*.mp4', recursive=True):
                s = (os.path.getsize(f), int(os.path.getmtime(f)))
                if seen.get(f) == ('done',) + s:
                    continue
                if seen.get(f) == s:
                    dd = SYNC_DST + f[len(SYNC_SRC):]
                    os.makedirs(os.path.dirname(dd), exist_ok=True)
                    shutil.copy(f, dd)
                    seen[f] = ('done',) + s
                else:
                    seen[f] = s
        except Exception:
            pass
        time.sleep(5)
threading.Thread(target=autosync, daemon=True).start()
subprocess.Popen(['python', 'main.py', '--listen', '127.0.0.1', '--port', '8188'], cwd='/content/ComfyUI', stdout=open('/content/comfy.log', 'w'), stderr=subprocess.STDOUT)
for _ in range(180):
    time.sleep(2)
    if 'To see the GUI' in open('/content/comfy.log').read():
        break
else:
    ng('ComfyUI が起動しない', 'このセルをもう一度押す。だめなら /content/comfy.log の最後を見る')
subprocess.Popen(['/content/cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8188'], stdout=open('/content/cloudflared.log', 'w'), stderr=subprocess.STDOUT)
url = None
for _ in range(60):
    time.sleep(2)
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('/content/cloudflared.log').read())
    if m:
        url = m.group(0); break
if url:
    ok('ComfyUI が起動した。下の URL を【コピー】して、新しいタブの【アドレス欄に貼って】開く（クリックすると断られる。人に教えない）')
    print('   ', url)
    print('   できた動画のコピー先：', SYNC_DST.replace('/content/drive/MyDrive/', 'マイドライブ/'))
else:
    ng('URL が出ない', 'このセルをもう一度押す')

## 6. ComfyUI で小さなテスト → アニメのカットを作る
1. ComfyUI の左のメニュー「ワークフロー」→ **test_t2v_5s** → 右上の「実行」。最初の1本はモデルを読み込むので少し長い。
2. 下のセル6を押すと、素材のフォルダ（カットごとに start.png・voice.wav・prompt.txt）から、カットごとのワークフローができる。
3. ComfyUI の画面を**再読み込み（F5）**すると、ワークフローの一覧に並ぶ → 開いて「実行」。

In [ ]:
# 6. 素材のフォルダから、カットごとのワークフローを作る
SET = 'sample'   # ← 素材の組の名前（自分の素材なら 'my_video' などに変える）
PROMPT = ''''''   # ← プロンプトをここに貼ってもよい（空なら各カットの prompt.txt を使う）
assert RESULT.get('ready'), 'NG: セル1が OK になっていない。セル1から順に押す'
exec(open('/content/h3_job_patch.py', encoding='utf-8').read())
IN = BASE + '/inputs/' + SET
TPL = open('/content/h3_i2v_lipsync.json', encoding='utf-8').read()
if not os.path.isdir(IN) or not any(os.path.isdir(f'{IN}/{j}') for j in os.listdir(IN)):
    os.makedirs(IN + '/c01', exist_ok=True)
jobs = sorted(j for j in os.listdir(IN) if os.path.isdir(f'{IN}/{j}'))
made = []
for JOB in jobs:
    d = f'{IN}/{JOB}'
    if PROMPT.strip():
        open(f'{d}/prompt.txt', 'w', encoding='utf-8').write(PROMPT.strip())
    try:
        d, cut = load_cut(IN, JOB)
        wf, info = patch(json.loads(TPL), d, cut, JOB, '/content/ComfyUI/input')
        json.dump(wf, open(f'/content/ComfyUI/user/default/workflows/{SET}_{JOB}.json', 'w', encoding='utf-8'), ensure_ascii=False)
        made.append(f"{SET}_{JOB}（{info['seconds']}秒）")
    except ValueError as e:
        ng(f'{JOB}: {e}', f'マイドライブ/AI_Colab/H3/inputs/{SET}/{JOB} に start.png・voice.wav・prompt.txt を入れて、このセルをもう一度')
if made:
    ok('ワークフローを作った：' + '、'.join(made) + '。ComfyUI の画面を再読み込み（F5）→ ワークフロー → 開いて「実行」')

## 9. 必ず：ランタイムを止める
できた動画がドライブ（マイドライブ/AI_Colab/H3/output）に入ったのを確かめてから押します。タブを閉じただけでは止まりません。

In [ ]:
# 9. ランタイムを止める（ComfyUI も URL も消える。ドライブの動画は残る）
from google.colab import runtime
runtime.unassign()